# Lab 13 — StatevectorEstimator y Cálculo de Energía Guiado

La primitiva `StatevectorEstimator` de Qiskit 2.x permite calcular valores esperados
⟨ψ|O|ψ⟩ para un observable O dado un estado cuántico |ψ⟩ preparado por un circuito.

**Aplicaciones**: energía VQE, correlaciones, magnetización, medición de observables en hardware.

## 1. StatevectorEstimator: API básica

La interfaz V2 usa la llamada `.run([(circuit, observable, params), ...])`. El resultado se accede como `result[0].data.evs` para el valor esperado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

# Circuito simple: estado de Bell
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)

# Observables
observables = {
    'XX': SparsePauliOp('XX'),
    'YY': SparsePauliOp('YY'),
    'ZZ': SparsePauliOp('ZZ'),
    'ZI': SparsePauliOp('ZI'),
    'IZ': SparsePauliOp('IZ'),
}

print('Valores esperados para el estado de Bell |Φ+⟩:')
for name, obs in observables.items():
    job = estimator.run([(qc_bell, obs)])
    val = job.result()[0].data.evs
    print(f'  ⟨{name}⟩ = {val:+.4f}')

## 2. Múltiples observables en una sola llamada

Qiskit 2.x permite pasar una lista de tuples (circuito, observable, params) en una sola llamada.
Esto es más eficiente que llamadas separadas cuando se miden múltiples observables del mismo estado.

In [ ]:
# Hamiltoniano de Heisenberg
H_heis = SparsePauliOp.from_list([('XX', 1.0), ('YY', 1.0), ('ZZ', 1.0)])

# Ansatz parametrizado
theta = ParameterVector('θ', 4)
ansatz = QuantumCircuit(2)
ansatz.ry(theta[0], 0)
ansatz.ry(theta[1], 1)
ansatz.cx(0, 1)
ansatz.ry(theta[2], 0)
ansatz.ry(theta[3], 1)

# Escaneo de energía para diferentes parámetros
angles = np.linspace(0, 2*np.pi, 50)
energies = []

# Batch: todos los puntos en una sola llamada
batch = [(ansatz, H_heis, [a, 0, 0, 0]) for a in angles]
job = estimator.run(batch)
energies = [job.result()[i].data.evs for i in range(len(angles))]

print(f'Min energía: {min(energies):.4f}')
print(f'Max energía: {max(energies):.4f}')
print(f'(Batch de {len(angles)} evaluaciones en 1 llamada)')

## 3. Medición de energía vs parámetro θ₀

Visualizamos cómo ⟨H⟩ varía al barrer θ₀ con el resto de parámetros fijos en 0.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(angles / np.pi, energies, 'b-', linewidth=2)
E_exact = np.linalg.eigvalsh(H_heis.to_matrix())[0]
ax.axhline(E_exact, color='red', linestyle='--', label=f'E₀ = {E_exact:.3f}')
ax.set_xlabel('θ₀ / π', fontsize=12)
ax.set_ylabel('⟨H_Heisenberg⟩', fontsize=12)
ax.set_title('Energía vs parámetro del ansatz', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Gradiente analítico (parameter shift rule)

Para calcular dE/dθ sin diferencias finitas, usamos el **parameter shift rule**:

$$\frac{\partial}{\partial \theta_k}\langle H\rangle = \frac{1}{2}\left[\langle H\rangle_{\theta_k+\pi/2} - \langle H\rangle_{\theta_k-\pi/2}\right]$$

Esto produce gradientes exactos con solo 2 evaluaciones por parámetro.

In [ ]:
def parameter_shift_gradient(ansatz, observable, params, idx):
    """Gradiente analítico para el parámetro params[idx]."""
    shift = np.pi / 2
    params_plus = params.copy()
    params_plus[idx] += shift
    params_minus = params.copy()
    params_minus[idx] -= shift

    job = estimator.run([
        (ansatz, observable, params_plus),
        (ansatz, observable, params_minus),
    ])
    e_plus = job.result()[0].data.evs
    e_minus = job.result()[1].data.evs
    return (e_plus - e_minus) / 2

# Gradiente en un punto
params0 = np.array([0.5, 0.3, -0.2, 0.8])
print('Gradiente ∂E/∂θ en', params0)
for k in range(len(params0)):
    grad_k = parameter_shift_gradient(ansatz, H_heis, params0, k)
    print(f'  ∂E/∂θ_{k} = {grad_k:+.6f}')

## 5. Estimator con Hamiltoniano de muchos cuerpos

Extendemos a 4 qubits con la cadena de Heisenberg:

$$H = \sum_{i=0}^{2}(X_i X_{i+1} + Y_i Y_{i+1} + Z_i Z_{i+1})$$

In [ ]:
# Heisenberg 4 qubits
n = 4
pauli_list = []
for i in range(n-1):
    for op in ['X', 'Y', 'Z']:
        ops = ['I'] * n
        ops[i] = op
        ops[i+1] = op
        pauli_list.append((''.join(reversed(ops)), 1.0))

H4 = SparsePauliOp.from_list(pauli_list)
E4_exact = np.linalg.eigvalsh(H4.to_matrix())[0]
print(f'Heisenberg 4q: E₀ = {E4_exact:.6f}')

# Ansatz para 4 qubits
th4 = ParameterVector('θ', 8)
a4 = QuantumCircuit(4)
for q in range(4): a4.ry(th4[q], q)
for q in range(3): a4.cx(q, q+1)
for q in range(4): a4.ry(th4[4+q], q)

# Calcular energía en punto aleatorio y en estado óptimo aproximado
np.random.seed(0)
rand_params = np.random.uniform(-np.pi, np.pi, 8)
job = estimator.run([(a4, H4, rand_params)])
E_rand = job.result()[0].data.evs
print(f'Energía en punto aleatorio: {E_rand:.4f}')
print(f'Energía exacta E₀:          {E4_exact:.4f}')
print(f'\nPrincipial variacional: {E_rand:.4f} ≥ {E4_exact:.4f} → {E_rand >= E4_exact}')